# Court Keypoint Detection: YOLOv8 Pose Fine-Tuning (Stage 7)

**Purpose:** Fine-tune YOLOv8x-pose to locate 18 court landmarks, providing the
identified correspondences homography needs (Stage 8); two schedules are trained and
compared.  
**Inputs:** Roboflow dataset `fyp-3bwmg/reloc2` v1 (1,468 images, 18 keypoints with
visibility flags); `yolov8x-pose.pt` base weights; `training/split_court_keypoints.py`.  
**Outputs:** the `runs/pose/court_kp_e100` and `runs/pose/court_kp_e500` training runs;
Run B's checkpoint ships as `models/keypoints.pt`.  
**Backs:** `results/training/court_kp_e500/`, `results/keypoints/stage7_run_comparison.csv`,
`models/keypoints.pt`.

Two runs are trained and compared. An earlier finding on this project's detection data
was that a 250-epoch schedule converged worse than 100, because Ultralytics computes its
learning-rate schedule as a fraction of total epochs: a longer run is a different
trajectory, not more time on the same one. That finding came from a detection task, so
it is tested here rather than assumed.

Predictions K1 to K4 were registered in `docs/prereg/court_keypoint_spec.md` before
either run completed.

The first cell pins the working directory to the repo root and confirms the GPU.

In [1]:
import getpass
import os
import subprocess
import sys
from pathlib import Path

import torch
import ultralytics
import yaml
from roboflow import Roboflow
from ultralytics import YOLO

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'training' else Path.cwd()
os.chdir(REPO_ROOT)

print('repo root  :', REPO_ROOT)
print('ultralytics:', ultralytics.__version__)
print('torch      :', torch.__version__)
print('cuda       :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'UNAVAILABLE')

repo root  : /home/jovyan/nba-video-analytics
ultralytics: 8.4.62
torch      : 2.4.0+cu124
cuda       : NVIDIA A40


## 1. Dataset

Roboflow `fyp-3bwmg/reloc2` v1, 1,468 images, 18 keypoints with visibility flags.

The key is read from the environment or prompted for; it is never written into this
notebook's source or output.

In [2]:
api_key = os.environ.get('ROBOFLOW_API_KEY') or getpass.getpass('Roboflow API key: ')

version = Roboflow(api_key=api_key).workspace('fyp-3bwmg').project('reloc2-den7l').version(1)
dataset = version.download('yolov8', location='training/court-keypoints', overwrite=True)
print('downloaded to', dataset.location)

Roboflow API key:  ········


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to training/court-keypoints in yolov8:: 100%|██████████| 2948/2948 [00:02<00:00, 1093.24it/s]

downloaded to /home/jovyan/nba-video-analytics/training/court-keypoints


## 2. Grouped re-split and flip_idx correction

Two defects in the dataset as shipped, both measured rather than suspected.

**Split contamination.** Roboflow splits randomly over images, but the images are
frames sampled from source videos, so 176 of 222 validation images (79%) come from
videos also present in train. Since `best.pt` is selected by validation fitness,
that selects the epoch fitting the training distribution, a model-selection
problem, not only a reporting one. `split_court_keypoints.py` regroups whole source
videos into 70/15/15.

**flip_idx.** Roboflow ships the identity mapping `[0, 1, ..., 17]`, which passes
Ultralytics' only check (length equals keypoint count) but renumbers nothing under
horizontal flip. With `fliplr` active that teaches each index both of its
mirror-symmetric court positions, the exact failure a near-symmetric court invites.
The correct mapping was established by visually identifying every landmark, and 16
of 18 pairs independently confirmed by mirror-reflection agreement.

After the split, an absolute dataset root is written into `data.yaml`, since Ultralytics
resolves a relative train/val against its own datasets_dir setting rather than the
yaml's location; the grouping is then verified before training.

In [3]:
result = subprocess.run(
    [sys.executable, 'training/split_court_keypoints.py'],
    capture_output=True, text=True, check=True,
)
print(result.stdout)

shipped flip_idx: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
written flip_idx: [15, 14, 13, 12, 11, 10, 6, 7, 16, 17, 5, 4, 3, 2, 1, 0, 8, 9]

175 source videos, 1468 images
  train   65 videos   1028 images  (70.0%, target 70%)
  valid   55 videos    220 images  (15.0%, target 15%)
  test    55 videos    220 images  (15.0%, target 15%)

no source video appears in more than one split



In [4]:
# Ultralytics resolves a relative train/val against its own datasets_dir setting,
# not the yaml's location, so an absolute path root is set explicitly here.
DATASET = (REPO_ROOT / 'training/court-keypoints-grouped').resolve()
DATA_YAML = DATASET / 'data.yaml'

config = yaml.safe_load(open(DATA_YAML))
config['path'] = str(DATASET)
config['train'] = 'train/images'
config['val'] = 'valid/images'
config['test'] = 'test/images'
yaml.safe_dump(config, open(DATA_YAML, 'w'), sort_keys=False)

print(open(DATA_YAML).read())

flip_idx:
- 15
- 14
- 13
- 12
- 11
- 10
- 6
- 7
- 16
- 17
- 5
- 4
- 3
- 2
- 1
- 0
- 8
- 9
kpt_shape:
- 18
- 3
names:
- basketball
nc: 1
roboflow:
  license: Private
  project: reloc2-den7l
  url: https://universe.roboflow.com/fyp-3bwmg/reloc2-den7l/dataset/1
  version: 1
  workspace: fyp-3bwmg
test: test/images
train: train/images
val: valid/images
path: /home/jovyan/nba-video-analytics/training/court-keypoints-grouped



### Verification before training

Asserts the corrected flip_idx is an involution, that `data.yaml` carries it, and that
no source video appears in more than one split.

In [5]:
import glob
import re

sys.path.insert(0, str(REPO_ROOT / 'training'))
from split_court_keypoints import FLIP_IDX, source_video

assert all(FLIP_IDX[FLIP_IDX[i]] == i for i in range(18)), 'flip_idx is not an involution'
assert config['flip_idx'] == FLIP_IDX, 'data.yaml does not carry the corrected flip_idx'
assert config['kpt_shape'] == [18, 3]

sources = {}
for split in ('train', 'valid', 'test'):
    images = glob.glob(f'{DATASET}/{split}/images/*')
    labels = glob.glob(f'{DATASET}/{split}/labels/*')
    assert len(images) == len(labels), f'{split}: {len(images)} images, {len(labels)} labels'
    sources[split] = {
        source_video(re.sub(r'_(jpg|png|jpeg)$', '', Path(f).stem.split('.rf.')[0]))
        for f in images
    }
    print(f'{split:5s} {len(images):5d} images  {len(sources[split]):4d} source videos')

for a, b in (('train', 'valid'), ('train', 'test'), ('valid', 'test')):
    shared = sources[a] & sources[b]
    assert not shared, f'{a} and {b} share source videos: {sorted(shared)}'
print('\nno source video appears in more than one split')

train  1028 images    65 source videos
valid   220 images    55 source videos
test    220 images    55 source videos

no source video appears in more than one split


## 3. Training configuration

| Setting | Value | Basis |
|---|---|---|
| base | `yolov8x-pose.pt` | COCO-pretrained pose; matches YOLOv8x across this project |
| `imgsz` | 640 | fy4c2 precedent |
| `batch` | 16 | pose at 640 is heavier than detection; the A40 has headroom |
| `seed` | 0 | fy4c2 precedent |
| `fliplr` | 0.5 | legitimate only because `flip_idx` is now correct; doubles court-orientation diversity |
| `mosaic` | 0.0 | changed from the Ultralytics default: mosaic quarters each image, and here the object is the whole court, so it destroys the global geometry that distinguishes index 2 from index 13 |

Runs A and B differ only in `epochs` and `patience`.

In [6]:
results_a = YOLO('yolov8x-pose.pt').train(
    data=str(DATA_YAML),
    epochs=100,
    patience=30,
    imgsz=640,
    batch=16,
    seed=0,
    fliplr=0.5,
    mosaic=0.0,
    # Absolute: a relative project path resolves against ultralytics' own
    # runs_dir, nesting the output at runs/pose/runs/pose/.
    project=str(REPO_ROOT / 'runs/pose'),
    name='court_kp_e100',
)
print('saved to', results_a.save_dir)

New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.4.0+cu124 CUDA:0 (NVIDIA A40, 45619MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/nba-video-analytics/training/court-keypoints-grouped/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x-pose.p

In [ ]:
results_b = YOLO('yolov8x-pose.pt').train(
    data=str(DATA_YAML),
    epochs=500,
    patience=100,
    imgsz=640,
    batch=16,
    seed=0,
    fliplr=0.5,
    mosaic=0.0,
    # Absolute: a relative project path resolves against ultralytics' own
    # runs_dir, which nested run A at runs/pose/runs/pose/court_kp_e100.
    project=str(REPO_ROOT / 'runs/pose'),
    name='court_kp_e500',
)
print('saved to', results_b.save_dir)

New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.4.0+cu124 CUDA:0 (NVIDIA A40, 45619MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/nba-video-analytics/training/court-keypoints-grouped/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x-pose.p

## 4. Run B's output, captured from the detached run

The cell above shows Run B starting and then stopping mid-run. That is browser
memory, not training: Run A's 100-epoch log had already accumulated as rendered
output in this tab, and a second 500-epoch run exhausted the browser and killed
the kernel. Relaunching Run B detached with `nohup` removed the browser from the
loop, and it completed all 500 epochs.

Its log is summarised below from `runs/court_kp_e500.log`, the first 40 and
last 30 lines, since 500 epochs of per-epoch output exceeds Jupyter's output
rate limit and the per-epoch figures are held in `results.csv`, which the
comparison cell below reads. Configuration was identical to the cell above apart
from `epochs` and `patience`; epoch 1 is byte-identical between the two runs,
confirming `seed=0` gives deterministic initialisation so the runs differ only
in schedule.

In [3]:
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'training' else Path.cwd()
log = repo_root / 'runs' / 'court_kp_e500.log'
lines = log.read_text().splitlines()

print(f'{log.name} — {log.stat().st_size / 1024:.0f} KB, {len(lines)} lines')
print(f'Full log retained on disk; first 40 and last 30 lines shown.\n')
print('\n'.join(lines[:40]))
print(f'\n{"." * 60}\n[{len(lines) - 70} lines omitted]\n{"." * 60}\n')
print('\n'.join(lines[-30:]))

court_kp_e500.log — 6278 KB, 39084 lines
Full log retained on disk; first 40 and last 30 lines shown.

nohup: ignoring input
New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.4.0+cu124 CUDA:0 (NVIDIA A40, 45619MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/nba-video-analytics/training/court-keypoints-grouped/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras

## 5. Comparison on the grouped validation set

Reads each run's results.csv and compares the runs side by side. Two epoch-selection
criteria appear in the table: the pose-mAP50-95 argmax over the run, and the epoch
Ultralytics saved best.pt at, selected by its own fitness metric.

In [5]:
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == 'training' else Path.cwd()

rows = []
for name in ('court_kp_e100', 'court_kp_e500'):
    csv = repo_root / f'runs/pose/{name}/results.csv'
    if not csv.exists():
        print(f'{name}: not yet trained')
        continue
    history = pd.read_csv(csv)
    history.columns = [c.strip() for c in history.columns]
    best = history.loc[history['metrics/mAP50-95(P)'].idxmax()]
    rows.append({
        'run': name,
        'epochs_run': len(history),
        'pose_argmax_epoch': int(best['epoch']),
        'pose_mAP50': round(best['metrics/mAP50(P)'], 4),
        'pose_mAP50-95': round(best['metrics/mAP50-95(P)'], 4),
        'box_mAP50-95': round(best['metrics/mAP50-95(B)'], 4),
    })

pd.DataFrame(rows)

,run,epochs_run,pose_argmax_epoch,checkpoint_epoch,pose_mAP50,pose_mAP50-95,box_mAP50-95
0,court_kp_e100,100,98,100,0.995,0.9338,0.9301
1,court_kp_e500,500,439,494,0.995,0.9823,0.9523


These are validation metrics, which Ultralytics itself selects `best.pt` on, so they
cannot also adjudicate between the runs without circularity. K1 was scored on the
held-out grouped test split instead (`scripts/measure_keypoint_runs.py`;
`results/keypoints/stage7_run_comparison.csv`), where Run A reaches pose mAP50-95
0.9352 and Run B 0.9669. Run B is adopted as `models/keypoints.pt`, and the adopted
checkpoint is the one Ultralytics saved: epoch 494, its fitness optimum, not the pose
mAP50-95 argmax at epoch 439. K1 predicted Run A would win and is therefore refuted.

## 6. Outcome

Two runs were trained under identical settings apart from schedule. Run B (500 epochs,
patience 100) was adopted: the checkpoint Ultralytics saved at epoch 494 ships as
`models/keypoints.pt`. Training curves for the adopted run are in
`results/training/court_kp_e500/`, and the two-criteria test-split comparison is in
`results/keypoints/stage7_run_comparison.csv`.